# Темы 10 и 12. Бенчмарк архитектур и дифференцированный подбор моделей под роли

**Модуль III · Пример 5 из 5**

Сводим предыдущие тетради в **измеримое сравнение**:
1. **Бенчмарк архитектур** — одиночный ReAct-агент vs иерархическая МАС vs МАС с критиком по качеству, числу вызовов модели, токенам и времени. Оценка качества — **объективная (fact-based)**: проверяем наличие ожидаемых фактов в ответе (без LLM-судьи; он — тема модуля 4).
2. **Дифференцированный подбор моделей под роли** (тема 12) — мощная модель для планировщика/критика, экономичная для типовых специалистов; сравнение с «единой моделью на всех ролях».

Эталонные реализации архитектур вынесены в модуль `mas_agents.py`.

In [1]:
import os, json, time
import pandas as pd
import mas_agents as A

# Два уровня моделей. Мощная — для рассуждения (планировщик/критик),
# лёгкая — для типовых специалистов. Провайдер отдаёт список моделей по
# GET /v1/models — оттуда выбираются доступные лёгкие модели (nano/mini/flash/lite).
STRONG = os.environ.get("LLM_MODEL_STRONG", os.environ.get("LLM_MODEL", "qwen/qwen3.7-max"))
LIGHT  = os.environ.get("LLM_MODEL_CHEAP",  "openai/gpt-4.1-nano")
print("STRONG:", STRONG, "| LIGHT:", LIGHT)

STRONG: qwen/qwen3.7-max | LIGHT: openai/gpt-4.1-nano


## Набор тестовых задач с эталонными фактами

Небольшой набор с **ожидаемыми фактами** для объективной проверки. Задачи мультидоменные — там, где МАС должна выигрывать за счёт явной декомпозиции.

In [2]:
TASKS = [
    {"query": "Можно ли вернуть заказ ORD-1001 и на какую сумму? Учти политику возврата.",
     "expected": ["4990", "14 дней"]},
    {"query": "Есть ли USB-C хаб в наличии и какой статус доставки у заказа ORD-1002?",
     "expected": ["USB-C", "CDEK"]},
]
for t in TASKS:
    print("-", t["query"][:70], "| факты:", t["expected"])

- Можно ли вернуть заказ ORD-1001 и на какую сумму? Учти политику возвра | факты: ['4990', '14 дней']
- Есть ли USB-C хаб в наличии и какой статус доставки у заказа ORD-1002? | факты: ['USB-C', 'CDEK']


## Бенчмарк 1: три архитектуры

Прогоняем каждую конфигурацию по всем задачам и собираем метрики. Ожидание: одиночный агент дешевле и быстрее, МАС полнее покрывает домены, критик добавляет задержку, но повышает надёжность.

In [3]:
def bench(configs, tasks):
    rows = []
    for name, fn in configs.items():
        for t in tasks:
            r = fn(t["query"])
            rows.append({"config": name, "quality": A.quality_score(r["answer"], t["expected"]),
                         "llm_calls": r["llm_calls"], "tokens": r["tokens"], "seconds": r["seconds"]})
    return pd.DataFrame(rows)

configs = {
    "single_light": lambda q: A.react_agent(q, model=LIGHT),
    "single_strong": lambda q: A.react_agent(q, model=STRONG),
    "hierarchical": lambda q: A.hierarchical_mas(q, STRONG, STRONG, STRONG),
    "mas_critic":   lambda q: A.hierarchical_mas(q, STRONG, STRONG, STRONG, critic_model=STRONG),
}
df = bench(configs, TASKS)
summary = df.groupby("config").mean(numeric_only=True).round(2)
print(summary.to_string())

               quality  llm_calls  tokens  seconds
config                                            
hierarchical      0.25        6.0  4722.0    58.61
mas_critic        0.75        7.5  6543.0    81.88
single_light      0.75        2.0   687.5     4.20
single_strong     0.75        2.0  1859.0    10.76


## Бенчмарк 2: дифференцированный подбор моделей под роли

Три варианта одной иерархической МАС на одной задаче:
- **all_strong** — все роли на мощной модели;
- **all_light** — все роли на лёгкой модели;
- **mixed** — мощная для планировщика и координатора, **лёгкая — для специалистов**.

Так виден компромисс: лёгкая модель дешевле и быстрее, смешанная схема экономит на типовых ролях, сохраняя качество там, где важно рассуждение.

In [4]:
task = TASKS[0]
variants = {
    "all_strong": A.hierarchical_mas(task["query"], STRONG, STRONG, STRONG),
    "all_light":  A.hierarchical_mas(task["query"], LIGHT, LIGHT, LIGHT),
    "mixed":      A.hierarchical_mas(task["query"], planner_model=STRONG,
                                     specialist_model=LIGHT, coordinator_model=STRONG),
}
rows = [{"variant": k, "quality": A.quality_score(v["answer"], task["expected"]),
         "llm_calls": v["llm_calls"], "tokens": v["tokens"], "seconds": v["seconds"]}
        for k, v in variants.items()]
print(pd.DataFrame(rows).to_string(index=False))

   variant  quality  llm_calls  tokens  seconds
all_strong      0.0          6    5182    65.36
 all_light      1.0          8    1865    14.74
     mixed      0.5          8    3697    50.86


## Итоги

- **Одиночный ReAct** — минимум вызовов и токенов; конкурентоспособен на простых задачах, но рискует не покрыть все домены сложного запроса.
- **Иерархическая МАС** — больше вызовов и токенов, но явная декомпозиция повышает полноту покрытия.
- **МАС + критик** — ещё дороже и медленнее, зато служит сеткой безопасности качества.
- **Дифференцированный подбор моделей под роли** позволяет снизить стоимость (экономичные специалисты), сохранив качество там, где важно рассуждение (мощные планировщик и критик). Это ключ к экономике краевого развёртывания (часть 2 практической работы).

**Наблюдение по реальному прогону (важно):** на данном наборе задач **лёгкая модель (gpt-4.1-nano)** как одиночный агент дала то же качество при кратно меньших стоимости и времени, а в бенчмарке 2 вариант `all_light` оказался и самым дешёвым, и лучшим по метрике. Вывод: превосходство мощной модели или МАС **нельзя постулировать** — его нужно измерять; часто лёгкой модели достаточно, а сложную архитектуру и мощные модели стоит подключать точечно (критик, сложные многодоменные задачи). Метрика здесь грубая (наличие фактов на малом наборе), поэтому важны размер набора и повторяемость.

**Общий вывод модуля:** архитектура выбирается под задачу и подкрепляется измерениями — по качеству, стоимости и времени, а не декларативно.